# Video 06 - Giving the model hands: your first tool

**You are here:** Video 6 of 13 - the Tools phase begins (see `../you-are-here.svg`). On the agent anatomy picture, today lights up **Tools**.

**The one idea:** the model **never runs your code**. It returns a tool call (structured output: which function, which arguments). **You** run it and send the result back.

**By the end you can:**
1. Describe a Python function as a tool.
2. Do one full tool round by hand.
3. Explain why tool names and descriptions matter so much.

**Before you start:** attach this notebook to **Serverless**.

In [0]:
import json
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"
OTHER_MODEL = "databricks-claude-haiku-4-5"
QUESTION = "What is 48729 * 3817?"
print("Connected. Correct answer (Python):", 48729 * 3817)

In [0]:
# Remember Video 01? Without a tool, the model guesses.
answer = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": QUESTION + " Answer with just the number."}]
).choices[0].message.content
print("Model alone:", answer, "| correct:", 48729 * 3817)

## Step 1 - A normal Python function
Nothing AI about it (diagram: `../diagrams/01-the-tool-call-dance.svg`).

In [0]:
def multiply(a: float, b: float) -> float:
    return a * b

print(multiply(48729, 3817))

## Step 2 - Describe the tool (the menu)
The model can't see your code. It only sees a **name**, a **description** (when to use it) and **parameters** - a JSON schema, like Video 05 (diagram: `../diagrams/02-function-to-schema.svg`).

In [0]:
MULTIPLY_TOOL = {
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers exactly. Use this for any multiplication.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "First number"},
                "b": {"type": "number", "description": "Second number"},
            },
            "required": ["a", "b"],
        },
    },
}

## Step 3 - The model asks for the tool
We send the question **and** the menu (`tools=[...]`).

**Predict first:** what will `message.content` be? And `finish_reason`?

In [0]:
messages = [{"role": "user", "content": QUESTION}]

response = client.chat.completions.create(model=MODEL, messages=messages, tools=[MULTIPLY_TOOL])
choice = response.choices[0]
msg = choice.message

print("finish_reason:", choice.finish_reason)      # 'tool_calls' (Video 03!)
print("content      :", msg.content)              # None - it isn't answering yet
call = msg.tool_calls[0]
print("wants to call:", call.function.name)
print("arguments    :", call.function.arguments, "<- a JSON string, type", type(call.function.arguments).__name__)

## Step 4 - YOU run the tool
Parse the arguments (`json.loads`, Video 05) and call your function.

In [0]:
args = json.loads(call.function.arguments)
result = multiply(**args)
print(f"We ran multiply({args}) -> {result}")

## Step 5 - Send the result back
Add **two** messages: the model's request, then your result with role `tool` and the **same** `tool_call_id`. Then call the model again. (The strict rules for these messages are Video 07.)

In [0]:
messages.append(msg.model_dump(exclude_none=True))     # the model's tool request
messages.append({
    "role": "tool",
    "tool_call_id": call.id,                           # links our result to its request
    "content": str(result),                            # tool results are sent as text
})

final = client.chat.completions.create(model=MODEL, messages=messages, tools=[MULTIPLY_TOOL])
print("finish_reason:", final.choices[0].finish_reason)
print("Final answer :", final.choices[0].message.content)

In [0]:
messages.append({"role": "assistant", "content": final.choices[0].message.content})
who = {"user": "the user", "assistant": "the model", "tool": "YOUR CODE"}

for i, m in enumerate(messages, 1):
    if m.get("tool_calls"):
        detail = f"tool_call -> {m['tool_calls'][0]['function']['name']}({m['tool_calls'][0]['function']['arguments']})"
    else:
        detail = m["content"]
    print(f"{i}. [{m['role']:9}] written by {who[m['role']]:10} | {detail[:70]}")

## The model decides when to use a tool
Tools are **options**, not orders.

**Predict first:** with the multiply tool attached, will the model call it for a geography question?

In [0]:
for model in [MODEL, OTHER_MODEL]:
    c = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": "What is the capital of France?"}], tools=[MULTIPLY_TOOL]
    ).choices[0]
    print(f"{model}: finish_reason={c.finish_reason!r}, tool called: {bool(c.message.tool_calls)} -> {(c.message.content or '')[:50]}")

## Cost meter: tools are not free
Tool descriptions are sent with **every** call - and models wrap them in extra instructions. And one tool round needs **two** model calls.

**Predict first:** how many input tokens does "Hi" cost with and without one small tool?

In [0]:
hi = [{"role": "user", "content": "Hi"}]
for model in [MODEL, OTHER_MODEL]:
    without = client.chat.completions.create(model=model, messages=hi, max_tokens=5).usage.prompt_tokens
    with_tool = client.chat.completions.create(model=model, messages=hi, tools=[MULTIPLY_TOOL], max_tokens=5).usage.prompt_tokens
    print(f"{model}: without tools {without:4} input tokens | with 1 tool {with_tool:4}")

print("\nOur tool round used 2 model calls:", response.usage.total_tokens, "+", final.usage.total_tokens,
      "=", response.usage.total_tokens + final.usage.total_tokens, "tokens")
# Lesson: only give the model the tools it actually needs.

## What could go wrong? Vague tool descriptions
The model only knows tools by **name and description** (diagram: `../diagrams/03-descriptions-matter.svg`). Two tools, both described as "Does stuff with two numbers". Secretly `tool_a` adds and `tool_b` multiplies.

**Predict first:** which one will the model pick for a multiplication?

In [0]:
def tool_spec(name, description):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": {"x": {"type": "number"}, "y": {"type": "number"}}, "required": ["x", "y"]}}}

def one_tool_round(question, tools, functions, model=MODEL):
    """Steps 3-5 from above in one helper: ask, run the requested tool, send the result, return the answer."""
    msgs = [{"role": "user", "content": question}]
    msg = client.chat.completions.create(model=model, messages=msgs, tools=tools).choices[0].message
    if not msg.tool_calls:
        return "(no tool used)", msg.content
    call = msg.tool_calls[0]
    result = functions[call.function.name](**json.loads(call.function.arguments))
    msgs += [msg.model_dump(exclude_none=True), {"role": "tool", "tool_call_id": call.id, "content": str(result)}]
    answer = client.chat.completions.create(model=model, messages=msgs, tools=tools).choices[0].message.content
    return call.function.name, answer

VAGUE_TOOLS = [tool_spec("tool_a", "Does stuff with two numbers."), tool_spec("tool_b", "Does stuff with two numbers.")]
VAGUE_FUNCTIONS = {"tool_a": lambda x, y: x + y, "tool_b": lambda x, y: x * y}   # tool_a adds, tool_b multiplies

for model in [MODEL, OTHER_MODEL]:
    picked, answer = one_tool_round(QUESTION, VAGUE_TOOLS, VAGUE_FUNCTIONS, model)
    print(f"{model}: picked {picked} -> {answer}")

In [0]:
CLEAR_TOOLS = [
    tool_spec("add", "Add two numbers exactly. Use this for addition only."),
    tool_spec("multiply", "Multiply two numbers exactly. Use this for any multiplication."),
]
CLEAR_FUNCTIONS = {"add": lambda x, y: x + y, "multiply": lambda x, y: x * y}

for model in [MODEL, OTHER_MODEL]:
    picked, answer = one_tool_round(QUESTION, CLEAR_TOOLS, CLEAR_FUNCTIONS, model)
    print(f"{model}: picked {picked} -> {answer}")
# Same functions. Only the names and descriptions changed.

## Recap
* **The model never runs your code.** It returns a `tool_call`; you run it; you send the result back.
* A tool = **name + description + parameters** (a JSON schema).
* One round: ask (`finish_reason == "tool_calls"`) -> run -> send `tool` message with the matching `tool_call_id` -> final answer.
* Tools are **options** - the model decides when to use them.
* Tools cost tokens on every call, and one round = two model calls.
* **Names and descriptions** are the model's only clue - make them clear.

## Practise
* [exercises.md](../exercises.md) - 3 exercises + bonus (about 15 minutes). Add your code below this cell.
* [quiz.md](../quiz.md) - 5 questions.
* Solutions: `06-first-tool-solutions` (try first!).

**Next:** Video 07 - The golden rules of tools.